In [2]:
import pandas as pd

print("Pandas imported successfully!")

Pandas imported successfully!


In [3]:
DATA_PATH = "../TMDB_data/cleaned/"

movies = pd.read_csv(DATA_PATH + "movies_cleaned.csv")

genres = pd.read_csv(DATA_PATH + "genres_cleaned.csv")

movie_genres = pd.read_csv(DATA_PATH + "movie_genres_cleaned.csv")

cast = pd.read_csv(DATA_PATH + "cast_cleaned.csv")

crew = pd.read_csv(DATA_PATH + "crew_cleaned.csv")

movie_keywords = pd.read_csv(DATA_PATH + "movie_keywords_cleaned.csv")

print("All 6 datasets loaded successfully!")

All 6 datasets loaded successfully!


In [4]:
datasets = {
    "movies": movies,
    "genres": genres,
    "movie_genres": movie_genres,
    "cast": cast,
    "crew": crew,
    "movie_keywords": movie_keywords
}

for name, df in datasets.items():
    print(f"{name}: {df.shape[0]} rows and {df.shape[1]} columns")

movies: 2503 rows and 13 columns
genres: 19 rows and 2 columns
movie_genres: 6878 rows and 2 columns
cast: 24902 rows and 6 columns
crew: 7289 rows and 6 columns
movie_keywords: 41681 rows and 3 columns


In [5]:
movies.head()

,movie_id,title,original_language,release_date,runtime,budget,revenue,popularity,vote_average,vote_count,status,overview,release_year
0,157336,Interstellar,en,2014-11-05,169,165000000.0,7.466067e+08,73.0452,8.484,40566,Released,The adventures of a group of explorers who mak...,2014
1,27205,Inception,en,2010-07-15,148,160000000.0,8.390306e+08,55.8497,8.400,39727,Released,"Cobb, a skilled thief who commits corporate es...",2010
2,24428,The Avengers,en,2012-04-25,143,220000000.0,1.518816e+09,72.2515,8.062,38993,Released,When an unexpected enemy emerges and threatens...,2012
3,155,The Dark Knight,en,2008-07-16,152,185000000.0,1.004558e+09,60.5510,8.534,36255,Released,Batman raises the stakes in his war on crime. ...,2008
4,19995,Avatar,en,2009-12-16,162,237000000.0,2.923706e+09,48.1996,7.610,34348,Released,"In the 22nd century, a paraplegic Marine is di...",2009


## Qusetion 1 - List the total number of unique movies, genres, cast members and crew members collected across all tables.

In [6]:
# Count unique movies
total_movies = movies["movie_id"].nunique()

# Count unique genres
total_genres = genres["genre_id"].nunique()

# Count unique cast members
total_cast = cast["person_id"].nunique()

# Count unique crew members
total_crew = crew["person_id"].nunique()

print("Total unique movies:", total_movies)
print("Total unique genres:", total_genres)
print("Total unique cast members:", total_cast)
print("Total unique crew members:", total_crew)

Total unique movies: 2503
Total unique genres: 19
Total unique cast members: 11300
Total unique crew members: 2848


## Question 2 - List all movies with missing or zero budget/revenue values, and decide how these should be treated for the rest of the analysis.

In [7]:
# Find movies with missing or zero budget/revenue

problem_movies = movies[
    (movies["budget"].isna()) |
    (movies["budget"] == 0) |
    (movies["revenue"].isna()) |
    (movies["revenue"] == 0)
]

print("Total movies with missing or zero budget/revenue:",
      len(problem_movies))

problem_movies[["title", "budget", "revenue"]].head(10)

Total movies with missing or zero budget/revenue: 133


,title,budget,revenue
324,Zack Snyder's Justice League,70000000.0,0.0
340,Bird Box,19800000.0,0.0
403,365 Days,0.0,9458590.0
468,To All the Boys I've Loved Before,0.0,0.0
554,Prey,65000000.0,0.0
598,The Kissing Booth,0.0,0.0
717,Enola Holmes,21000000.0,0.0
723,Extraction,65000000.0,0.0
754,"Love, Rosie",0.0,4439431.0
842,Bright,90000000.0,0.0


In [8]:
# Budget issues
budget_missing = movies["budget"].isna().sum()
budget_zero = (movies["budget"] == 0).sum()

# Revenue issues
revenue_missing = movies["revenue"].isna().sum()
revenue_zero = (movies["revenue"] == 0).sum()

print("Budget missing:", budget_missing)
print("Budget zero:", budget_zero)

print("Revenue missing:", revenue_missing)
print("Revenue zero:", revenue_zero)

Budget missing: 0
Budget zero: 90
Revenue missing: 0
Revenue zero: 102


In [9]:
# Movies with valid budget
movies_valid_budget = movies[
    movies["budget"].notna() & (movies["budget"] > 0)
]

# Movies with valid revenue
movies_valid_revenue = movies[
    movies["revenue"].notna() & (movies["revenue"] > 0)
]

# Movies with both valid budget and revenue
movies_valid_financial = movies[
    (movies["budget"] > 0) &
    (movies["revenue"] > 0)
]

In [10]:
print("Original movies:", len(movies))

print("Valid budget movies:", len(movies_valid_budget))

print("Valid revenue movies:", len(movies_valid_revenue))

print("Valid financial movies:", len(movies_valid_financial))

Original movies: 2503
Valid budget movies: 2413
Valid revenue movies: 2401
Valid financial movies: 2370


## Question 3 - List the profit (revenue − budget) and return on investment for every movie that has valid budget and revenue figures.

In [11]:
#We already created this DataFrame in Question 2:
# 
movies_roi = movies_valid_financial.copy()

In [12]:
# Calculate profit
movies_roi["profit"] = (
    movies_roi["revenue"] - movies_roi["budget"]
)

# Calculate ROI percentage
movies_roi["ROI"] = (
    movies_roi["profit"] / movies_roi["budget"]
) * 100

In [13]:
movies_roi[
    ["title", "budget", "revenue", "profit", "ROI"]
].head(10)

,title,budget,revenue,profit,ROI
0,Interstellar,165000000.0,7.466067e+08,5.816067e+08,352.488913
1,Inception,160000000.0,8.390306e+08,6.790306e+08,424.394144
2,The Avengers,220000000.0,1.518816e+09,1.298816e+09,590.370689
3,The Dark Knight,185000000.0,1.004558e+09,8.195584e+08,443.004564
4,Avatar,237000000.0,2.923706e+09,2.686706e+09,1133.631235
5,Deadpool,58000000.0,7.828373e+08,7.248373e+08,1249.719564
6,Fight Club,63000000.0,1.008538e+08,3.785375e+07,60.085322
7,Avengers: Infinity War,300000000.0,2.052415e+09,1.752415e+09,584.138346
8,The Shawshank Redemption,25000000.0,2.834147e+07,3.341469e+06,13.365876
9,Pulp Fiction,8000000.0,2.139288e+08,2.059288e+08,2574.109525


In [14]:
top_roi_movies = movies_roi.sort_values(
    by="ROI",
    ascending=False
)

top_roi_movies[
    ["title", "budget", "revenue", "profit", "ROI"]
].head(10)

,title,budget,revenue,profit,ROI
929,Paranormal Activity,15000.0,193355800.0,193340800.0,1.288939e+06
921,The Blair Witch Project,60000.0,248639099.0,248579099.0,4.142985e+05
1315,Obsession,750000.0,474895195.0,474145195.0,6.321936e+04
836,Bambi,858000.0,267447150.0,266589150.0,3.107100e+04
1031,Mad Max,350000.0,100000000.0,99650000.0,2.847143e+04
2031,Night of the Living Dead,114000.0,30236452.0,30122452.0,2.642320e+04
1445,The Texas Chain Saw Massacre,140000.0,30922680.0,30782680.0,2.198763e+04
807,Halloween,325000.0,70260597.0,69935597.0,2.151865e+04
1767,Dragon Ball Super: Broly,1000000.0,125002821.0,124002821.0,1.240028e+04
535,Snow White and the Seven Dwarfs,1488423.0,184925486.0,183437063.0,1.232426e+04


## Question 4 - List the top 15 movies by return on investment, considering only movies with a budget of at least $1 million.

In [15]:
movies_min_budget = movies_roi[
    movies_roi["budget"] >= 1000000
]

print("Movies with budget >= $1 million:", len(movies_min_budget))

Movies with budget >= $1 million: 2327


In [16]:
top_15_roi = movies_min_budget.sort_values(
    by="ROI",
    ascending=False
).head(15)

top_15_roi[
    ["title", "budget", "revenue", "profit", "ROI"]
]

,title,budget,revenue,profit,ROI
1767,Dragon Ball Super: Broly,1000000.0,125002821.0,124002821.0,12400.282100
535,Snow White and the Seven Dwarfs,1488423.0,184925486.0,183437063.0,12324.256142
1791,The Rocky Horror Picture Show,1400000.0,171181400.0,169781400.0,12127.242857
467,Rocky,1000000.0,117253345.0,116253345.0,11625.334500
1227,Gone with the Wind,4000000.0,402352579.0,398352579.0,9958.814475
716,The Jungle Book,4000000.0,378000000.0,374000000.0,9350.000000
635,Cinderella,2900000.0,263600000.0,260700000.0,8989.655172
372,Saw,1200000.0,104045735.0,102845735.0,8570.477917
719,One Hundred and One Dalmatians,3600000.0,303000000.0,299400000.0,8316.666667
263,E.T. the Extra-Terrestrial,10500000.0,797307407.0,786807407.0,7493.403876


## Question 5 - List the number of movies released per year, and identify which year had the highest movie output in the dataset.

In [ ]:
# Count movies released per year
movies_per_year = movies.groupby("release_year")["movie_id"].count()

# Find the year with the highest movie output
highest_year = movies_per_year.idxmax()
highest_count = movies_per_year.max()


print("Year with highest movie output:", highest_year)
print("Number of movies released:", highest_count)

movies_per_year

Year with highest movie output: 2016
Number of movies released: 121


release_year
1921     1
1922     1
1927     1
1931     2
1936     1
        ..
2022    65
2023    62
2024    43
2025    26
2026     8
Name: movie_id, Length: 89, dtype: int64

## Question 6 - List all movies whose runtime is a statistical outlier compared to the rest of the dataset.

In [22]:
#Calculate Q1, Q3 and IQR

# Calculate quartiles
Q1 = movies["runtime"].quantile(0.25)
Q3 = movies["runtime"].quantile(0.75)

# Calculate IQR
IQR = Q3 - Q1

# Calculate outlier boundaries
lower_bound = Q1 - (1.5 * IQR)
upper_bound = Q3 + (1.5 * IQR)

print("Q1:", Q1)
print("Q3:", Q3)
print("IQR:", IQR)
print("Lower boundary:", lower_bound)
print("Upper boundary:", upper_bound)

Q1: 99.0
Q3: 126.0
IQR: 27.0
Lower boundary: 58.5
Upper boundary: 166.5


In [23]:
runtime_outliers = movies[
    (movies["runtime"] < lower_bound) |
    (movies["runtime"] > upper_bound)
]

print("Total runtime outliers:", len(runtime_outliers))

runtime_outliers[
    ["title", "runtime", "release_year"]
].sort_values(by="runtime", ascending=False)

Total runtime outliers: 60


,title,runtime,release_year
324,Zack Snyder's Justice League,242,2021
1227,Gone with the Wind,238,1939
829,Once Upon a Time in America,229,1984
1610,Lawrence of Arabia,228,1962
1774,Ben-Hur,212,1959
587,The Irishman,209,2019
1277,Seven Samurai,207,1954
1304,Killers of the Flower Moon,206,2023
187,The Godfather Part II,202,1974
20,The Lord of the Rings: The Return of the King,201,2003


## Question 7 - List each movie's primary genre (the first genre listed) and compare average popularity across primary genres.

In [24]:
print(movie_genres.head(10))

   movie_id  genre_id
0    157336        12
1    157336        18
2    157336       878
3     27205        28
4     27205       878
5     27205        12
6     24428       878
7     24428        28
8     24428        12
9       155        28


In [ ]:
print(genres.head(10))

   genre_id   genre_name
0        28       Action
1        12    Adventure
2        16    Animation
3        35       Comedy
4        80        Crime
5        99  Documentary
6        18        Drama
7     10751       Family
8        14      Fantasy
9        36      History


In [26]:
#create a column that records the order of each genre within a movie.

movie_genres_ordered = movie_genres.copy()

movie_genres_ordered["genre_order"] = (
    movie_genres_ordered.groupby("movie_id").cumcount()
)

movie_genres_ordered.head(10)

,movie_id,genre_id,genre_order
0,157336,12,0
1,157336,18,1
2,157336,878,2
3,27205,28,0
4,27205,878,1
5,27205,12,2
6,24428,878,0
7,24428,28,1
8,24428,12,2
9,155,28,0


In [27]:
#Combine the three DataFrames

movie_genre_data = movies.merge(
    movie_genres_ordered,
    on="movie_id",
    how="inner"
)

movie_genre_data = movie_genre_data.merge(
    genres,
    on="genre_id",
    how="left"
)

movie_genre_data.head(10)

,movie_id,title,original_language,release_date,runtime,budget,revenue,popularity,vote_average,vote_count,status,overview,release_year,genre_id,genre_order,genre_name
0,157336,Interstellar,en,2014-11-05,169,165000000.0,7.466067e+08,73.0452,8.484,40566,Released,The adventures of a group of explorers who mak...,2014,12,0,Adventure
1,157336,Interstellar,en,2014-11-05,169,165000000.0,7.466067e+08,73.0452,8.484,40566,Released,The adventures of a group of explorers who mak...,2014,18,1,Drama
2,157336,Interstellar,en,2014-11-05,169,165000000.0,7.466067e+08,73.0452,8.484,40566,Released,The adventures of a group of explorers who mak...,2014,878,2,Science Fiction
3,27205,Inception,en,2010-07-15,148,160000000.0,8.390306e+08,55.8497,8.400,39727,Released,"Cobb, a skilled thief who commits corporate es...",2010,28,0,Action
4,27205,Inception,en,2010-07-15,148,160000000.0,8.390306e+08,55.8497,8.400,39727,Released,"Cobb, a skilled thief who commits corporate es...",2010,878,1,Science Fiction
5,27205,Inception,en,2010-07-15,148,160000000.0,8.390306e+08,55.8497,8.400,39727,Released,"Cobb, a skilled thief who commits corporate es...",2010,12,2,Adventure
6,24428,The Avengers,en,2012-04-25,143,220000000.0,1.518816e+09,72.2515,8.062,38993,Released,When an unexpected enemy emerges and threatens...,2012,878,0,Science Fiction
7,24428,The Avengers,en,2012-04-25,143,220000000.0,1.518816e+09,72.2515,8.062,38993,Released,When an unexpected enemy emerges and threatens...,2012,28,1,Action
8,24428,The Avengers,en,2012-04-25,143,220000000.0,1.518816e+09,72.2515,8.062,38993,Released,When an unexpected enemy emerges and threatens...,2012,12,2,Adventure
9,155,The Dark Knight,en,2008-07-16,152,185000000.0,1.004558e+09,60.5510,8.534,36255,Released,Batman raises the stakes in his war on crime. ...,2008,28,0,Action


In [28]:
#Select each movie's primary genre

primary_genre_movies = (
    movie_genre_data[
        movie_genre_data["genre_order"] == 0
    ]
    .copy()
)

primary_genre_movies[
    ["title", "genre_name", "popularity"]
].head(10)

,title,genre_name,popularity
0,Interstellar,Adventure,73.0452
3,Inception,Action,55.8497
6,The Avengers,Science Fiction,72.2515
9,The Dark Knight,Action,60.5510
12,Avatar,Action,48.1996
15,Deadpool,Action,29.3387
18,Fight Club,Drama,53.7525
20,Avengers: Infinity War,Adventure,81.4079
23,The Shawshank Redemption,Drama,76.9826
25,Pulp Fiction,Thriller,42.0350


In [29]:
#Compare average popularity by primary genre
genre_popularity = (
    primary_genre_movies
    .groupby("genre_name")["popularity"]
    .mean()
    .sort_values(ascending=False)
)

genre_popularity

genre_name
Science Fiction    21.261836
Music              18.324340
Animation          16.811212
Family             16.710635
Action             16.385188
War                15.470317
Horror             14.851835
Adventure          14.785510
Fantasy            13.022920
Mystery            12.427131
Western            12.322680
Thriller           11.294211
Drama              11.278534
Crime              10.739498
History            10.414567
Comedy              9.984262
Romance             9.976708
Documentary         7.726200
Name: popularity, dtype: float64

## Question 8 - List the top 10 most prolific actors by number of movies, along with their average movie rating across those movies.

In [30]:
cast[["person_id", "actor_name", "movie_id"]].head(10)

,person_id,actor_name,movie_id
0,10297,Matthew McConaughey,157336
1,1813,Anne Hathaway,157336
2,3895,Michael Caine,157336
3,83002,Jessica Chastain,157336
4,1893,Casey Affleck,157336
5,8210,Wes Bentley,157336
6,17052,Topher Grace,157336
7,851784,Mackenzie Foy,157336
8,9560,Ellen Burstyn,157336
9,12074,John Lithgow,157336


In [31]:
movies[["movie_id", "title", "vote_average"]].head(10)

,movie_id,title,vote_average
0,157336,Interstellar,8.484
1,27205,Inception,8.400
2,24428,The Avengers,8.062
3,155,The Dark Knight,8.534
4,19995,Avatar,7.610
5,293660,Deadpool,7.625
6,550,Fight Club,8.438
7,299536,Avengers: Infinity War,8.240
8,278,The Shawshank Redemption,8.727
9,680,Pulp Fiction,8.482


In [33]:
actor_movies = cast[
    ["person_id", "actor_name", "movie_id"]
].drop_duplicates()

actor_movies = actor_movies.merge(
    movies[["movie_id", "vote_average"]],
    on="movie_id",
    how="inner"
)

actor_movies.head(10)

,person_id,actor_name,movie_id,vote_average
0,10297,Matthew McConaughey,157336,8.484
1,1813,Anne Hathaway,157336,8.484
2,3895,Michael Caine,157336,8.484
3,83002,Jessica Chastain,157336,8.484
4,1893,Casey Affleck,157336,8.484
5,8210,Wes Bentley,157336,8.484
6,17052,Topher Grace,157336,8.484
7,851784,Mackenzie Foy,157336,8.484
8,9560,Ellen Burstyn,157336,8.484
9,12074,John Lithgow,157336,8.484


In [34]:
actor_analysis = (
    actor_movies
    .groupby(["person_id", "actor_name"])
    .agg(
        total_movies=("movie_id", "nunique"),
        average_rating=("vote_average", "mean")
    )
    .reset_index()
)

actor_analysis.head()

,person_id,actor_name,total_movies,average_rating
0,2,Mark Hamill,7,7.582000
1,3,Harrison Ford,21,7.217095
2,4,Carrie Fisher,7,7.452857
3,5,Peter Cushing,1,8.206000
4,6,Anthony Daniels,7,7.367571


In [35]:
top_10_actors = actor_analysis.sort_values(
    by=["total_movies", "average_rating"],
    ascending=[False, False]
).head(10)

top_10_actors

,person_id,actor_name,total_movies,average_rating
679,2231,Samuel L. Jackson,39,6.938692
106,287,Brad Pitt,38,7.257289
127,380,Robert De Niro,38,7.228316
37,85,Johnny Depp,37,6.856973
13,31,Tom Hanks,33,7.289758
1272,5293,Willem Dafoe,32,7.114781
413,1245,Scarlett Johansson,32,7.056344
2674,13240,Mark Wahlberg,32,6.630063
161,500,Tom Cruise,31,7.040613
64,192,Morgan Freeman,31,6.963742


## Question 9 - List any movie titles that appear more than once in the dataset, along with their release years, to check for duplicates or remakes.

In [ ]:
#Find movie titles that appear more than once
# Count how many times each movie title appears
title_counts = movies["title"].value_counts()

# Select titles appearing more than once
duplicate_titles = title_counts[title_counts > 1]

print("Number of repeated movie titles:", len(duplicate_titles))

Number of repeated movie titles: 49


In [37]:
#Display titles with their release years
repeated_movies = movies[
    movies["title"].isin(duplicate_titles.index)
]

repeated_movies = repeated_movies[
    ["title", "release_year"]
].sort_values(
    by=["title", "release_year"]
)

repeated_movies

,title,release_year
878,A Nightmare on Elm Street,1984
1928,A Nightmare on Elm Street,2010
261,Aladdin,1992
339,Aladdin,2019
792,Alice in Wonderland,1951
...,...,...
1737,The Thing,2011
744,Total Recall,1990
860,Total Recall,2012
2500,Transformers,1987


In [40]:
same_title_year = movies[
    movies.duplicated(
        subset=["title", "release_year"],
        keep=False
    )
].sort_values(
    by=["title", "release_year"]
)

print("Number of repeated movie titles:", len(duplicate_titles))
print("Records with repeated title and year:", len(same_title_year))

#same_title_year[["title", "release_year"]]

Number of repeated movie titles: 49
Records with repeated title and year: 0


## Question 10- List movies whose popularity score and vote count don't follow the general pattern between the two — i.e., unusually high  popularity with a low vote count, or vice versa.

In [41]:
movies[["popularity", "vote_count"]].describe()

,popularity,vote_count
count,2503.000000,2503.000000
mean,13.691796,6193.834998
std,18.109296,4959.007777
min,0.017000,475.000000
25%,7.450200,3027.000000
50%,10.397100,4361.000000
75%,14.866200,7276.000000
max,438.475700,40566.000000


In [42]:
movies["popularity"].corr(movies["vote_count"])

np.float64(0.36085002849007386)

In [ ]:
#High popularity + low vote count.
#Low popularity + high vote count.

In [ ]:
#calculate the thresholds directly from DataFrame
# Calculate thresholds
high_popularity = movies["popularity"].quantile(0.75)
low_popularity = movies["popularity"].quantile(0.25)

high_votes = movies["vote_count"].quantile(0.75)
low_votes = movies["vote_count"].quantile(0.25)

In [44]:
#Find Case A – High popularity, low vote count
high_pop_low_votes = movies[
    (movies["popularity"] > high_popularity) &
    (movies["vote_count"] < low_votes)
].copy()

high_pop_low_votes["observation"] = "High popularity, low votes"

print("High popularity, low votes:", len(high_pop_low_votes))

High popularity, low votes: 35


In [45]:
#Find Case B – Low popularity, high vote count
low_pop_high_votes = movies[
    (movies["popularity"] < low_popularity) &
    (movies["vote_count"] > high_votes)
].copy()

low_pop_high_votes["observation"] = "Low popularity, high votes"

print("Low popularity, high votes:", len(low_pop_high_votes))

Low popularity, high votes: 33


In [48]:
#Combine both results
unusual_movies = pd.concat(
    [high_pop_low_votes, low_pop_high_votes]
)

final_result = unusual_movies[
    ["title", "popularity", "vote_count", "observation"]
].sort_values(
    by="popularity",
    ascending=False
)

final_result

,title,popularity,vote_count,observation
2388,Disclosure Day,394.3346,2328,"High popularity, low votes"
2248,Backrooms,258.4743,2483,"High popularity, low votes"
2452,Mortal Kombat II,99.5231,2260,"High popularity, low votes"
2295,Lee Cronin's The Mummy,79.4601,2430,"High popularity, low votes"
2022,The Housemaid,41.4337,2789,"High popularity, low votes"
...,...,...,...,...
205,X-Men: First Class,1.2614,13680,"Low popularity, high votes"
354,X-Men: The Last Stand,1.1649,10511,"Low popularity, high votes"
322,X2,0.8734,11013,"Low popularity, high votes"
417,The Lost World: Jurassic Park,0.8620,9356,"Low popularity, high votes"
